# Model Comparison: ROC AUC for Every Experiment

**Why another score?** Macro F1 judges the **final category** each model picks. But every model first gives a **probability for each category** and then picks the highest one. Two models can pick the same category while one of them was much surer for the right reasons. **ROC AUC judges those probabilities.**

**The mental model: a ranking test.** Take one day that really was Severe and one day that was not. Ask the model: "which of these two days is more likely to be Severe?" **AUC is how often it ranks the real Severe day higher**, over all such pairs.

| AUC | Meaning |
|---|---|
| 0.5 | A coin toss: the probabilities say nothing |
| 0.7–0.8 | Useful |
| 0.8–0.9 | Good |
| above 0.9 | Very good ranking |
| 1.0 | Every Severe day ranked above every other day |

**Why it matters for us:**
- AUC does **not** depend on a threshold. Macro F1 uses "pick the highest probability"; AUC judges the probabilities **before** that choice. A high Severe AUC means a good **Severe threshold** exists, which is the next planned step.
- It is computed **one category against the rest** (6 scores), plus their average (**macro AUC**) and the **Dangerous** alert (probability of Very Poor + Severe).

**Persistence has no probabilities**: it says 100% for today's category and 0% for the rest. So it gets one point on the ROC curve instead of a full curve, and its AUC is only (recall + share of other days correctly left out) / 2. It is shown as a reference, but **only the learned models can be compared fairly with each other**.

**This notebook** loads every saved experiment, in the order we built them, and shows **where each step improved** (section 4), on 2023 and, like every decision now, on **five years** (section 6).

Run `lr_train_baseline.ipynb`, `dt_train.ipynb`, `rf_train.ipynb` and `xgb_train.ipynb` first: this notebook only loads their saved models.

## 1. Setup

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from sklearn.base import clone
from sklearn.metrics import f1_score, roc_curve

import sys
sys.path.insert(0, '../scripts')                 # utility.py lives in scripts/
from utility import (AQI_CATEGORIES, DANGEROUS, prepare_dataset, split_train_val_test, add_lr_features,
                     proba_table, persistence_proba, auc_scores, rolling_validation, rolling_summary)

DATA_PATH = '../data/processed/aqi_weather_2015_2025.csv'
MODEL_DIR = Path('models')
TARGET = 'tomorrow_status'
TREND = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']
pd.set_option('display.max_columns', 30)

## 2. The same 2023 days as every model

In [ ]:
df = prepare_dataset(DATA_PATH)
df = df[df[TREND].notna().all(axis=1)]                   # the same 195 rows every model drops
df_train, df_val, df_test = split_train_val_test(df)
y_val = df_val[TARGET].astype(str).to_numpy()
print('validation (2023):', len(df_val), 'days |', (y_val == 'Severe').sum(), 'Severe |',
      np.isin(y_val, DANGEROUS).sum(), 'Dangerous')

## 3. Load every experiment

Each saved bundle holds the model and the list of features it needs. Logistic regression also needs its own transforms (`add_lr_features`, with the city pressure normals saved in its bundle); the tree models use the raw columns.

In [ ]:
def lr_records(bundle, part):
    features = sum(bundle['features'].values(), [])
    return add_lr_features(part, pd.Series(bundle['city_pressure']))[features].to_dict(orient='records')

def tree_records(bundle, part):
    features = sum(bundle['features'].values(), [])
    return part[features].to_dict(orient='records')

STEPS = [                                                  # (name, saved file, how to build its features)
    ('1. logistic regression, balanced',   'lr_baseline.bin',   lr_records),
    ('2. logistic regression, no weights', 'lr_no_weights.bin', lr_records),
    ('3. logistic regression, tuned C',    'lr_best.bin',       lr_records),
    ('4. decision tree',                   'dt_best.bin',       tree_records),
    ('5. random forest',                   'rf_best.bin',       tree_records),
    ('6. xgboost',                         'xgb_best.bin',      tree_records),
]
missing = [file for _, file, _ in STEPS if not (MODEL_DIR / file).exists()]
assert not missing, f'run the training notebooks first, missing: {missing}'

BUNDLES, PROBA = {}, {'0. persistence': persistence_proba(df_val['status_today'])}
for name, file, records in STEPS:
    with open(MODEL_DIR / file, 'rb') as f:
        BUNDLES[name] = pickle.load(f)
    PROBA[name] = proba_table(BUNDLES[name]['model'], records(BUNDLES[name], df_val))

# Check: picking the most likely category gives back each model's known 2023 macro F1
pd.Series({name: f1_score(y_val, p.idxmax(axis=1), labels=AQI_CATEGORIES, average='macro')
           for name, p in PROBA.items()}, name='macro F1 (2023)').round(3).to_frame()

## 4. ROC AUC on 2023, and where each step improved

**AUC for every category, the average of the six (`macro`) and the Dangerous alert:**

In [ ]:
auc = pd.DataFrame({name: auc_scores(y_val, p) for name, p in PROBA.items()}).T
auc.round(3)

**What each step added:** every row is that experiment's AUC **minus the step before it** (positive = improved). Step 1 is compared with persistence.

In [ ]:
change = auc.diff().iloc[1:]
change.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
limit = np.abs(change.values[1:]).max()                  # scale on the steps between learned models; the jump
image = ax.imshow(change.values, cmap='RdBu',              # from persistence (step 1) is far larger and just saturates
                  norm=TwoSlopeNorm(0, -limit, limit), aspect='auto')
ax.set_xticks(range(change.shape[1]), change.columns, rotation=20)
ax.set_yticks(range(change.shape[0]), [f'{name}  (vs step before)' for name in change.index])
for i in range(change.shape[0]):
    for j in range(change.shape[1]):
        value = change.values[i, j]
        ax.text(j, i, f'{value:+.3f}', ha='center', va='center', fontsize=9,
                color='white' if abs(value) > 0.6 * limit else 'black')
ax.set_title('Change in ROC AUC from each experiment (blue = improved, red = worse)')
fig.colorbar(image, ax=ax, label='AUC change')
plt.tight_layout()
plt.show()

**Findings:**
- **The big jump is from persistence to any learned model** (macro AUC 0.793 → 0.930). Most of it comes from having probabilities at all: persistence can only say yes or no. The fair comparison is between the learned models.
- **All learned models rank the days well, and close together:** macro AUC 0.930 to 0.945. Between experiments, AUC moves by at most 0.017; macro F1 moved far more (0.598 to 0.679).
- **Step 2, removing class weights: macro AUC +0.009, but Severe unchanged (−0.001).** The weights barely changed how well the days are **ordered**; they changed **where the bar is set** for saying "Severe" (the "lowered bar" from `lr_train_baseline.ipynb`). That is why macro F1 jumped (0.598 → 0.647) while AUC hardly moved.
- **Step 3, tuning C: no change at all.** Tuning chose C = 1, the default, so it is the same model as step 2.
- **Step 4, the decision tree: the best Severe ranking (0.986, +0.011) but worse on Very Poor (−0.009) and Dangerous (−0.007).** Every day in a tree leaf (at least 100 days) gets the same probabilities, so a tree ranks days only coarsely. Overall, no gain (macro −0.000).
- **Step 5, the random forest: better on 7 of the 8 columns** (macro +0.004). Only Severe falls back (0.976, the same as logistic regression).
- **Step 6, XGBoost: small gains on almost every column,** and the **best macro AUC** (0.945).
- **Macro F1 and AUC disagree on the top two.** The forest has the better macro F1 (0.679 against 0.668); XGBoost has the better AUC (0.945 against 0.943). XGBoost **ranks** days slightly better; the forest **picks** better categories in the end, helped by its balanced weights, which move the bar towards the rare categories.

## 5. ROC curves: Severe and Dangerous

A ROC curve shows every possible threshold at once. Going right means more false alarms (share of other days wrongly flagged); going up means more real days caught. **The higher and further left the curve, the better.** Persistence is a single point, because it has only one setting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (title, truth, score) in zip(axes, [
        ('Severe', y_val == 'Severe', lambda p: p['Severe']),
        ('Dangerous (Very Poor + Severe)', np.isin(y_val, DANGEROUS), lambda p: p[DANGEROUS].sum(axis=1))]):
    for name, p in PROBA.items():
        if name.startswith('0.'):
            continue
        false_alarm_rate, caught_rate, _ = roc_curve(truth, score(p))
        ax.plot(false_alarm_rate, caught_rate, label=f"{name} ({auc.loc[name, title.split(' ')[0]]:.3f})")
    pred = score(PROBA['0. persistence']) > 0.5
    ax.scatter(pred[~truth].mean(), pred[truth].mean(), color='black', zorder=5, label='persistence (one point)')
    ax.plot([0, 1], [0, 1], color='grey', linestyle=':', linewidth=1)
    ax.set_xlim(0, 0.3)                                    # the useful part: few false alarms
    ax.set_title(f'{title}: ROC curves on 2023')
    ax.set_xlabel('false alarm rate (share of other days flagged)')
    ax.set_ylabel('share of real days caught')
    ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

**Findings:**
- **Severe: at persistence's own false-alarm level, the models do not catch more.** Persistence raised 25 false alarms in 2023 (0.5% of the other days) and caught 53% of the Severe days. At the same false-alarm level, the models catch 45% to 57%.
- **The models' advantage on Severe is choice.** Accept about 50 false alarms in the year (1% of other days) and the forest catches 67% and XGBoost 69%; accept about 100 (2%) and they catch 78% to 82%. Persistence has no such setting. With only 51 Severe days, every step in these curves is a single day.
- **Dangerous: a real gain at no extra cost.** At persistence's false-alarm level (103 false alarms, 2.2% of other days), the forest and XGBoost catch **82%** of the Dangerous days against persistence's **75%** (logistic regression 80%, the tree 73%).
- **So the Severe threshold step is a decision about how many false alarms are acceptable for each extra Severe day caught,** not a purely technical one.

## 6. Five years: does the AUC ranking hold?

The same rule as before: one year can mislead. Every experiment is retrained in each round of rolling validation (learn from the years before, score 2019 to 2023) with exactly its saved settings (`clone` copies the settings without the learned trees or weights). `rolling_validation` now also records macro AUC whenever a model gives probabilities. About 3 minutes.

In [ ]:
df_dev = pd.concat([df_train, df_val])                   # 2015-2023 only; the test years stay locked

def lr_xy_for(bundle):
    features = sum(bundle['features'].values(), [])
    def xy(train, val):
        city_pressure = train.groupby('city')['pressure_mean'].mean()     # from this round's training years only
        return (add_lr_features(train, city_pressure)[features].to_dict(orient='records'),
                add_lr_features(val, city_pressure)[features].to_dict(orient='records'))
    return xy

def tree_xy_for(bundle):
    return lambda train, val: (tree_records(bundle, train), tree_records(bundle, val))

rolling = pd.concat({
    name: rolling_validation(df_dev, lambda b=BUNDLES[name]: clone(b['model']),
                             (lr_xy_for if records is lr_records else tree_xy_for)(BUNDLES[name]))
    for name, _, records in STEPS}, names=['model']).reset_index(level=0)

auc_by_year = rolling.pivot_table(index='model', columns='val_year', values='macro_auc')
persistence_auc = rolling.groupby('val_year')['persistence_macro_auc'].first().rename('0. persistence')
auc_by_year = pd.concat([persistence_auc.to_frame().T, auc_by_year])
print('Macro AUC per year:')
auc_by_year.round(3)

**Five-year summary:** mean macro AUC next to mean macro F1 and the bar from before (`passes_bar` is still judged on macro F1):

In [ ]:
five_years = rolling_summary(rolling)
five_years['auc_change_vs_step_before'] = five_years['mean_macro_auc'].diff()
five_years[['mean_macro_auc', 'auc_change_vs_step_before', 'mean_macro_f1', 'mean_vs_persistence',
            'years_won', 'passes_bar']].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name in auc_by_year.index:
    if name.startswith(('0.', '3.')):                    # persistence (0.77-0.79) would squash the plot;
        continue                                           # step 3 is the same model as step 2
    ax.plot(auc_by_year.columns, auc_by_year.loc[name], marker='o', label=name)
ax.set_xticks(list(auc_by_year.columns))
ax.set_xlabel('validation year (learned from all earlier years)')
ax.set_ylabel('macro ROC AUC')
ax.set_title('Rolling validation: macro AUC of the learned models (persistence: 0.77-0.79)')
ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

**Findings:**
- **The ranking holds over five years.** XGBoost has the highest macro AUC in **every one** of the five years (mean **0.942**), and the random forest is second in every year (mean **0.940**).
- **The forest is the biggest step between learned models** (+0.010 over the tree's five-year average). On 2023 alone it looked like +0.004.
- **The decision tree ranks below the unweighted logistic regression** (0.930 against 0.935), the opposite of macro F1 (tree 0.631, logistic regression 0.613). Again: its probabilities are coarse. Only the balanced logistic regression ranks lower (0.929).
- **Removing class weights helps logistic regression's ranking in every year** (+0.006 on average), the same as on 2023.
- **AUC is steadier than macro F1.** Every learned model's AUC moves by about 0.02 or less between years, and XGBoost and the forest are the top two in every year. Macro F1 depends more on where the bar happens to fall in each year.

## Discussion

**Where each experiment improved** (five-year averages, each compared with the step before):

| Step | Macro F1 | Macro AUC | Where it helped, where it did not |
|---|---|---|---|
| 0. Persistence | 0.634 | 0.782 | Reference only: no probabilities |
| 1. Logistic regression, balanced | −0.020 | +0.146 | Every category ranked far better than yes/no; but too many rare forecasts |
| 2. Logistic regression, no weights | −0.001 | +0.006 | Middle categories (Moderate, Poor); Severe ranking unchanged: weights only move the bar |
| 3. Logistic regression, tuned C | 0.000 | 0.000 | No change: the tuned C is the default |
| 4. Decision tree | +0.018 | −0.005 | Best Severe ranking on 2023; coarse probabilities elsewhere |
| 5. Random forest | **+0.033** | **+0.010** | The biggest step: better in almost every category; passes the bar |
| 6. XGBoost | −0.009 | +0.002 | Best ranking in every year, but slightly worse final picks than the forest |

1. **Both scores agree on the leaders:** the random forest and XGBoost are the two best models, on 2023 and over five years.
2. **They disagree on the order.** XGBoost ranks the days slightly better (AUC); the forest picks the final category better (macro F1). The gap is small in both cases.
3. **All learned models rank days well (AUC above 0.93); the bigger differences come from where the bar is set.** Class weights changed macro F1 a lot and AUC hardly at all. So the next step, **tuning the Severe threshold**, can matter more than another model type. It should be tried on **both** the forest and XGBoost, and judged on five years.
4. **For Severe, the models catch more days only by accepting more false alarms.** At persistence's false-alarm level, no model catches more Severe days. For the **Dangerous** alert, the forest and XGBoost catch more (82% against 75%) **with the same number of false alarms**.
5. **For management:** the forecast already gives a better health warning than "tomorrow = today" at no extra cost. For the rarest category, Severe, the team has to decide how many false alarms are acceptable for each extra Severe day caught.